# GLOBAL CONSTANTS

In [ ]:
# Modify here
project_name = "NferenceInternalWorkingProject"
# project_name = "NSCLCWorkingProject"
version_of_work = "V8"
version_of_data = "V8"

## Don't modify below
basename_fetchdata_folder = f"/data/{project_name}/shared/Nbs_{version_of_work}/{version_of_data}/fetch_data/"
DATASET = "AMC1DataNSCLC"
TABLE_NAME  = f"stage_iii_nsclc_cohort_{version_of_work}"
SCRATCH_DB_NAME = "scratch_db." + TABLE_NAME
cohort_name_version_name_append = f"_{version_of_work}"
sys_path_of_cohortkit_package = f"/data/NSCLCWorkingProject/shared/Nbs_{version_of_work}/"
TABLE_NAME_2 = "study_cohort_with_min_diagnosis_date"
SCRATCH_DB_NAME_WITH_STUDY_DTM = f"scratch_db.{TABLE_NAME_2}"

dicom_study_description_mapping_file_path = f'/data/NferenceInternalWorkingProject/shared/Nbs_{version_of_work}/{version_of_data}/input/Mayo_Radiology_Description_Mapping.tsv'

print(f"""
================ PROJECT CONFIGURATION ================

Project Name                     : {project_name}
Version of Work                  : {version_of_work}

Base Folder                      : {basename_fetchdata_folder}
Sys Path of cohortkit package    : {sys_path_of_cohortkit_package}

Dataset                          : {DATASET}
Table Name                       : {TABLE_NAME}
Scratch DB Name                  : {SCRATCH_DB_NAME}
Cohort Version Suffix            : {cohort_name_version_name_append}

Dicom study description mapping  : {dicom_study_description_mapping_file_path}


=======================================================
""")

# Importing Important Libraries

In [ ]:
import nferhub
import pandas as pd
import os
import numpy as np
import re

In [ ]:
cohort_client = nferhub.client("cohort", dataset=DATASET)

In [ ]:
sql_client = nferhub.client("sql_blaze")
sql_client.connect(dataset = "AMC1DataNSCLC",schema_model = "source",data_version = "6.027")

# Importing CSVs

In [ ]:
fact_syn_dicom_radiology_df = pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/V8/process_data/FACT_SYN_DICOM_RADIOLOGY.csv', dtype = str)
radiology_dicom_df = pd.read_csv("/data/NferenceInternalWorkingProject/shared/Nbs_V8/V8/process_data/RADIOLOGY_DICOM.csv", dtype=str)

mayo_cohort_df =  pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/mayo_cohort_20260616.csv')
radiology_annotations_df1 = pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/DR.IMRAN - RECIST 1.1 ANNOTATION-Table1 (12).csv' , dtype= str)
radiology_annotations_df2 = pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/DR.SAI - RECIST 1.1 ANNOTATION-Table1 (12).csv' , dtype= str)

In [ ]:
radiology_annotations_df = pd.concat([radiology_annotations_df1, radiology_annotations_df2], axis=0)

In [ ]:
radiology_annotations_df.shape

In [ ]:
radiology_annotations_df['PATIENT ID'].unique()

In [ ]:
# passed_patients_df = pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/RECIST_FINAL_OUTPUT/quality_check_pipeline/recist_qc_modular_package/real_qc_outputs_v3/passed_patients_final.csv')
# previous_patients_df = pd.read_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/RECIST_FINAL_OUTPUT/OUTPUT_CSVs/zipping/TASK_DETAILS_V8.csv')


In [ ]:
# passed_patients_df.info()

In [ ]:
#previous_patients_df.PERSON_ID.nunique()

In [ ]:
# passed_patients_df.PERSON_ID.nunique()

In [ ]:
# passed_patients_list = passed_patients_df['PERSON_ID'].astype(str).drop_duplicates().tolist()
#previous_patients_list = previous_patients_df['PERSON_ID'].astype(str).drop_duplicates().tolist()

In [ ]:
# # Find elements in passed but not in previous
# new_passed_patients = list(set(passed_patients_list) - set(previous_patients_list))

# print(f"Found {len(new_passed_patients)} new passed patients.")
# print(new_passed_patients)

In [ ]:
# new_passed_patients = passed_patients_df[:130]

In [ ]:
# total_pids_list = passed_patients_list[:130] # + previous_patients_list

In [ ]:
# len(total_pids_list)

In [ ]:
# type(total_pids_list)

In [ ]:
radiology_annotations_df['PATIENT ID'].nunique()

In [ ]:
# len(list(set(new_pids)))

In [ ]:
# len(total_pids)

In [ ]:
# len(radiology_annotations_df['PATIENT ID'].unique())

In [ ]:
pd.set_option("display.max_colwidth", None)  
pd.set_option("display.max_columns", None)   
pd.set_option("display.width", 0) 

In [ ]:
import glob
import os
import pandas as pd
from pathlib import Path

# 1. Define the explicit folder paths based on your layout
# Starting from your current notebook folder ('.')
# v1_path = os.path.join(".", "raw_download", "V1", "dicom_api")
# v2_path = os.path.join(".", "raw_download", "V2", "dicom_api")
# v3_path = os.path.join(".", "raw_download", "V3", "dicom_api")
# v4_path = os.path.join(".", "raw_download", "V4", "dicom_api")
# v5_path = os.path.join(".", "raw_download", "V5", "dicom_api")
# v6_path = os.path.join(".", "raw_download", "V6", "dicom_api")

v7_path = os.path.join(".", "raw_download", "V7", "dicom_api")

# 2. Gather all CSV files from both directories recursively (looking inside subfolders like 'csvs')
# v1_csv_files = glob.glob(os.path.join(v1_path, "**", "*.csv"), recursive=True)
# v2_csv_files = glob.glob(os.path.join(v2_path, "**", "*.csv"), recursive=True)
# v3_csv_files = glob.glob(os.path.join(v3_path, "**", "*.csv"), recursive=True)
# v4_csv_files = glob.glob(os.path.join(v4_path, "**", "*.csv"), recursive=True)
# v5_csv_files = glob.glob(os.path.join(v5_path, "**", "*.csv"), recursive=True)
# v6_csv_files = glob.glob(os.path.join(v6_path, "**", "*.csv"), recursive=True)

v7_csv_files = glob.glob(os.path.join(v7_path, "**", "*.csv"), recursive=True)

# Combine both file lists together
# v1_csv_files + v2_csv_files + v3_csv_files + v4_csv_files + v5_csv_files + v6_csv_files
all_csv_files = v7_csv_files

# print(f"Found {len(v1_csv_files)} CSV files in V1.")
# print(f"Found {len(v2_csv_files)} CSV files in V2.")
# print(f"Found {len(v3_csv_files)} CSV files in V3.")
# print(f"Found {len(v4_csv_files)} CSV files in V4.")
# print(f"Found {len(v5_csv_files)} CSV files in V5.")
# print(f"Found {len(v6_csv_files)} CSV files in V6.")
print(f"Found {len(v7_csv_files)} CSV files in V7.")
print(f"Total CSV files found: {len(all_csv_files)}")

all_dfs = []
for file_path in all_csv_files:
    try:
        df = pd.read_csv(file_path, dtype = str)
        df["source_file"] = os.path.basename(file_path)

        # Dynamic version capture: Grabs whatever folder name is after 'raw_downloads'
        path_parts = Path(file_path).parts
        if "raw_download" in path_parts:
            idx = path_parts.index("raw_download")
            df["version"] = path_parts[idx + 1]  # Automatically captures V1, V2, V3, etc.
        else:
            df["version"] = "Unknown"

        all_dfs.append(df)
    except Exception as e:
        print(f"Skipping error file {file_path}: {e}")

# Combined master dataframe
master_df = (
    pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
)

In [ ]:
# # 1. Define the folder path ('.' means the current folder where your notebook is)
# folder_path = './' 


# csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# print(f"Found {len(csv_files)} CSV files.")
# print(csv_files)

In [ ]:
# df_list = []

# for file in csv_files:
#     file_path = os.path.join(folder_path, file)
    
    
#     temp_df = pd.read_csv(file_path, dtype = str)
#     df_list.append(temp_df)

# # # 4. Concatenate all dataframes into one single "Master" dataframe
# # # ignore_index=True ensures the row numbers (0, 1, 2...) are continuous
# master_df = pd.concat(df_list, ignore_index=True)

print(f"Master DataFrame created with {len(master_df)} total rows.")

In [ ]:
master_df.head()

In [ ]:
master_df = master_df.drop(columns = ['source_file', 'version'])

In [ ]:
master_df.duplicated().sum()

In [ ]:
master_df['Label'].unique()

In [ ]:
master_df = master_df.drop_duplicates().reset_index(drop = True)

In [ ]:
master_df.info()

In [ ]:
master_df['StudyDate'] = pd.to_datetime(master_df['StudyDate']).dt.date

In [ ]:
master_df.drop(columns = ['PhysicalCoordinates' , 'Modality'], inplace = True)

In [ ]:
unique_ids = tuple(master_df['PERSON_ID'].unique())
mapping_query = f"""
    SELECT PATIENT_CLINIC_NUMBER, NFER_PID 
    FROM DIM_PATIENT 
    WHERE PATIENT_CLINIC_NUMBER IN {unique_ids}
"""
mapping_df = sql_client.query(mapping_query)

In [ ]:
id_map = dict(zip(mapping_df['PATIENT_CLINIC_NUMBER'], mapping_df['NFER_PID']))
master_df['PERSON_ID'] = master_df['PERSON_ID'].map(id_map)

In [ ]:
master_df.dropna(subset = ['PERSON_ID'], inplace  = True)
master_df.sort_values(by = [ 'PERSON_ID' , 'descriptions'], inplace = True)
master_df = master_df.reset_index(drop = True)
master_df.head()

In [ ]:
master_df = master_df[master_df['descriptions']!= 'TEST']

In [ ]:
final_pids = (
    master_df["PERSON_ID"]
    .dropna()
    .drop_duplicates()
    .astype(str)
    .tolist()
)

In [ ]:
len(final_pids)

In [ ]:
master_df.descriptions.unique()

In [ ]:
import re

# 1. Flexible Followup Pattern
# Matches any sequence starting with F/FLL/FOL/FOLL, containing optional brackets/spaces, ending in digits
flexible_followup_pattern = r"^F[O0LWIU\s\[\]]*P*\s*\[?(\d+)"

# 2. Flexible Baseline Pattern 
# Matches BASELINE, BASELEINE, BASELINE1, etc.
flexible_baseline_pattern = r"^BAS[ELIN]*\d*"


master_df["descriptions"] = (
    master_df["descriptions"]
    .astype(str)
    .str.strip()
    .str.upper()
    # Normalize all baseline variations -> BASELINE
    .str.replace(flexible_baseline_pattern, "BASELINE", regex=True)
    # Normalize all typo followups -> FOLLOWUP<number>
    .str.replace(flexible_followup_pattern, r"FOLLOWUP\1", regex=True)
)

print(master_df["descriptions"].value_counts(dropna=False))

In [ ]:
no_list= ['ERROR', 'BASELEINE','IGNORE' , 'error', 'ERRORS']

In [ ]:
master_df = master_df[~master_df['descriptions'].isin(no_list)]

In [ ]:
master_df['PERSON_ID'] = master_df['PERSON_ID'].astype(str)

In [ ]:
master_df

In [ ]:
pid_list = master_df['PERSON_ID'].unique().tolist()
fact_syn_dicom_radiology_df = fact_syn_dicom_radiology_df[fact_syn_dicom_radiology_df['PERSON_ID'].isin(pid_list)]
fact_syn_dicom_radiology_df.head()

In [ ]:
fact_syn_dicom_radiology_df.info()

In [ ]:
fact_syn_dicom_radiology_df.rename(columns = {  'STUDY_INSTANCE_UID': 'StudyUID'  ,
                                                'SERIES_INSTANCE_UID' : 'SeriesUID' ,
                                                'SOPINSTANCE_UID': 'InstanceUID', 
                                                 'INSTANCE_NUMBER': 'Slice_number'}, inplace = True)

In [ ]:
master_df.info()

In [ ]:
master_df = pd.merge(master_df , fact_syn_dicom_radiology_df[['PERSON_ID', 'StudyUID', 'SeriesUID' ,
                                                                   'InstanceUID', 'Slice_number']] ,
                                on = ['PERSON_ID', 'StudyUID', 'SeriesUID' , 'InstanceUID'] ,
                         how = 'left')

In [ ]:
master_df.head()

In [ ]:
master_df.info()

In [ ]:
import ast

def add_z_coordinate(row):
    coords = row['PixelCoordinates']
    
    # Convert string "[152, 326, ...]" to actual list
    if isinstance(coords, str):
        coords = ast.literal_eval(coords)
    
    z = row['Slice_number']
    
    return [[coords[i], coords[i+1], z] for i in range(0, len(coords), 2)]


master_df['PixelCoordinates'] = master_df.apply(add_z_coordinate, axis=1)

In [ ]:
master_df.drop(columns = ['Slice_number'], inplace = True)

In [ ]:
master_df.descriptions.value_counts(dropna = False)

In [ ]:
master_df.PERSON_ID.unique()

In [ ]:
master_df.info()

In [ ]:
# columns_to_check = [col for col in master_df.columns if col != 'PixelCoordinates']

# # 2. Drop duplicates based ONLY on those columns
# master_df = master_df.drop_duplicates(subset=columns_to_check).reset_index(drop=True).copy()

In [ ]:
master_df = master_df[master_df['PERSON_ID'].isin(final_pids)]

In [ ]:
# len(passed_patients_list)

# Grist Sheet

In [ ]:
radiology_annotations_df.head(20)

In [ ]:
import numpy as np
import pandas as pd

def forward_fill_dataset_ids(df):
    s = df['PATIENT ID'].astype(str)
    extracted = s.str.extract(r'dataset_(.+)')[0]
    extracted_ids = np.where(s.str.contains('dataset_', na=False), extracted, np.nan)
    df['PATIENT ID'] = pd.Series(extracted_ids, index=df.index).ffill()
    return df

In [ ]:
radiology_annotations_df = forward_fill_dataset_ids(radiology_annotations_df)

In [ ]:
radiology_annotations_df.head(40)

In [ ]:
len(radiology_annotations_df['PATIENT ID'].unique())

In [ ]:
radiology_annotations_df['PATIENT ID'] = radiology_annotations_df['PATIENT ID'].astype(str).str.split('_').str[-1]

In [ ]:
len(radiology_annotations_df['PATIENT ID'].unique())

In [ ]:
radiology_annotations_df.head()

In [ ]:
len(final_pids)

In [ ]:
# shared_elements = set(final_pids).intersection(tf)

# # 2. Get the count
# count = len(shared_elements)
# print(f"Number of identical unique elements: {count}")
# print(f"The shared elements are: {shared_elements}")

In [ ]:
radiology_annotations_df.info()

In [ ]:
for x in radiology_annotations_df['PATIENT ID'].drop_duplicates().astype(str).tolist():
    if x not in final_pids:
        print(x)

In [ ]:
# total_pids_list

In [ ]:
# list_of_pids = []
radiology_annotations_df = radiology_annotations_df[radiology_annotations_df['PATIENT ID'].isin(final_pids)]

In [ ]:
radiology_annotations_df

In [ ]:
# radiology_annotations_df.dropna(subset = ['SERIES TYPE', 'LESION ID' , 'LESION DESCRIPTION' , 'LOCATION'  ] , inplace = True)

In [ ]:
radiology_annotations_df.info()

In [ ]:
radiology_annotations_df.dropna( subset =['PATIENT ID'] , inplace = True)
radiology_annotations_df['PATIENT ID'] = radiology_annotations_df['PATIENT ID'].astype(int)
radiology_annotations_df.rename( columns = {'PATIENT ID' : 'PERSON_ID' , 
                                            'SERIES TYPE' : 'SERIES_TYPE' , 
                                           'LESION DESCRIPTION' : 'LESION_DESCRIPTION' ,
                                            'LN_STATION' : 'STATION' ,
                                            'LESION ID' : 'LESION_ID' ,
                                           'NTL RESPONSE' : 'NTL_RESPONSE', 
                                            'NL TUMOR STATUS' : 'NL_TUMOR_STATUS',
                                           'NL RESPONSE' : 'NL_RESPONSE',
                                            'TL RESPONSE CAT' : "TL_RESPONSE_CAT" ,
                                            'FAILURE TYPE' : "FAILURE_TYPE" 
                                                } , inplace = True)

In [ ]:
radiology_annotations_df.head()

In [ ]:
radiology_annotations_df.PERSON_ID.nunique()

In [ ]:
master_df.rename( columns = { 'descriptions' : 'SERIES_TYPE' , 
                                     'Label' : 'LESION_ID' } , inplace = True)

In [ ]:
radiology_annotations_df

In [ ]:
radiology_annotations_df.PERSON_ID.nunique()

In [ ]:

radiology_annotations_df = radiology_annotations_df.dropna(subset=radiology_annotations_df.columns.difference(['PERSON_ID']), how='all')

In [ ]:
radiology_annotations_df

In [ ]:
radiology_annotations_df["SERIES_TYPE"] = radiology_annotations_df["SERIES_TYPE"].replace(["NaN", "nan", ""], np.nan)

# 2. Forward-fill the values downward until a new value appears
radiology_annotations_df["SERIES_TYPE"] = radiology_annotations_df["SERIES_TYPE"].ffill()

In [ ]:
radiology_annotations_df.info()

In [ ]:
radiology_annotations_df.head(20)

In [ ]:
radiology_annotations_df

In [ ]:
radiology_annotations_df['SERIES_TYPE'].value_counts(dropna=False)

In [ ]:
radiology_annotations_df['SERIES_TYPE'] = radiology_annotations_df['SERIES_TYPE'].str.strip().str.upper().replace(flexible_followup_pattern, r'FOLLOWUP\1', regex=False)
radiology_annotations_df['SERIES_TYPE'].value_counts(dropna=False)

In [ ]:
master_df.head(15)

In [ ]:
master_df['PERSON_ID'] = master_df['PERSON_ID'].astype(int)

In [ ]:
master_df_pre_final = pd.merge(master_df , radiology_annotations_df ,
             on = ['PERSON_ID' , 'SERIES_TYPE' , 'LESION_ID' ] , how = 'outer')

In [ ]:
master_df_pre_final.head(20)

In [ ]:
master_df_pre_final.info()

In [ ]:
master_df_pre_final[['PERSON_ID','StudyDate','SERIES_TYPE' ,'LESION_ID', 'LOCATION', 'LN STATION' , 'LATERALITY']].head(15)

In [ ]:
columns_to_fill = [
    'StudyDate', 
    'StudyUID', 
    'SeriesUID', 
    'Tagger'
      # or whatever your exact column name is for followups
]

# Ensure the columns actually exist in the DataFrame before filling to avoid KeyError
columns_to_fill = [col for col in columns_to_fill if col in master_df_pre_final.columns]

# 3. Group by patient and followup window, then fill missing values forward and backward
master_df_pre_final[columns_to_fill] = (
    master_df_pre_final
    .groupby(['PERSON_ID', 'SERIES_TYPE'])[columns_to_fill]
    .ffill()
    .bfill()
)

# 4. View the cleaned results
print(master_df_pre_final.info())

In [ ]:
master_df_pre_final['SERIES_TYPE'].value_counts(dropna=False)

In [ ]:
master_df_pre_final.LESION_ID.value_counts(dropna = False)

In [ ]:
un_wanted_list = ['TEST', 'FOLLOWUP25', 'FOLLOWUP3']

In [ ]:
master_df_pre_final = master_df_pre_final[master_df_pre_final['LESION_ID'] != 'TEST']


In [ ]:
master_df_pre_final = master_df_pre_final[~master_df_pre_final['LESION_ID'].isin(un_wanted_list)]

In [ ]:
# 1. Clean up the text by removing hidden spaces and forcing uppercase
master_df_pre_final['LESION_ID'] = master_df_pre_final['LESION_ID'].astype(str).str.strip().str.upper()

# 2. Define the pattern to allow TL, NTL, or NL followed by optional numbers
valid_pattern = r'^(TL|NTL|NL)\d*$'

# 3. Filter the DataFrame
master_df_pre_final = master_df_pre_final[
    master_df_pre_final['LESION_ID'].str.contains(valid_pattern, na=False, regex=True)
]

In [ ]:
master_df_pre_final.LESION_ID.value_counts(dropna = False)

In [ ]:
master_df_pre_final.dropna(subset = ['SERIES_TYPE', 'LESION_ID'], inplace = True)
master_df_pre_final = master_df_pre_final.reset_index(drop =True)

In [ ]:
# master_df_pre_final[master_df_pre_final['LESION_ID'] == 'NL1']

In [ ]:
master_df_pre_final

In [ ]:
master_df_pre_final.info()

In [ ]:
# 1. Define the tracking columns you want to fill down
tracking_cols = ['LATERALITY', 'LOCATION', 'LN STATION', 'LESION_DESCRIPTION']
tracking_cols = [col for col in tracking_cols if col in master_df_pre_final.columns]

# 2. Group by patient and lesion, then forward fill the tracking characteristics downward
# This ensures that once NL1 or NL2 gets its location, it carries over to future followups
master_df_pre_final[tracking_cols] = (
    master_df_pre_final
    .groupby(['PERSON_ID', 'LESION_ID'])[tracking_cols]
    .ffill()
)

print(master_df_pre_final[['PERSON_ID', 'SERIES_TYPE', 'LESION_ID'] + tracking_cols].head(20))

In [ ]:
# tracking_cols = [ 'LATERALITY', 'LOCATION' , 'LN STATION']
# # Make sure they exist in your dataframe
# tracking_cols = [col for col in tracking_cols if col in master_df_pre_final.columns]

# # 3. Create a clean lookup table using ONLY valid baseline rows
# baseline_lookup = (
#     master_df_pre_final[master_df_pre_final['SERIES_TYPE'] == 'BASELINE']
    
#     [['PERSON_ID','LESION_ID'] + tracking_cols]
# )


# master_df_cleaned = master_df_pre_final.drop(columns=tracking_cols)


# master_df_pre_final = pd.merge(
#     master_df_cleaned,
#     baseline_lookup,
#     on=['PERSON_ID', 'LESION_ID'],
#     how='left'
# )


# print(master_df_pre_final[['PERSON_ID', 'SERIES_TYPE', 'LESION_ID'] + tracking_cols].head(20))

In [ ]:
master_df_pre_final.head()

In [ ]:
master_df_pre_final.NL_RESPONSE.value_counts(dropna = False)

In [ ]:
master_df_pre_final.LESION_DESCRIPTION.value_counts(dropna = False)

In [ ]:
master_df_pre_final.head()

In [ ]:
master_df_pre_final

#### Radiology Dicom

In [ ]:
radiology_dicom_df.rename(columns = { 'SERIES_INSTANCE_UID': 'SeriesUID' ,'STUDY_INSTANCE_UID': 'StudyUID' }, inplace = True)

In [ ]:
radiology_dicom_df.head()

In [ ]:
master_df_pre_final.info()

In [ ]:
radiology_dicom_df['PERSON_ID'] = radiology_dicom_df['PERSON_ID'].astype(int)

In [ ]:
master_df_final = pd.merge( master_df_pre_final , radiology_dicom_df[['PERSON_ID', 'STUDY_DESCRIPTION', 'MODALITY_DESCRIPTION', 
                                                                   'BODY_PART', 'HARMONIZED_STUDY_DESCRIPTION',
                                                                   'StudyUID', 'SeriesUID']], 
                             on = ['PERSON_ID' ,  'StudyUID', 'SeriesUID'   ]  , how = 'left')

In [ ]:
master_df_final.info()

In [ ]:
master_df_final

In [ ]:
master_df_final.shape

#### Task Details

In [ ]:
task_details_df = master_df_final.rename( columns = {
                                                    'StudyUID' : 'study_id' ,
                                                     'SeriesUID' : 'series_id' ,
                                                    
                                                      'LESION_ID' : 'Lesion_id'  ,
                                                       'SERIES_TYPE' : 'series_type' ,
                                                         'PixelCoordinates' : 'Lesion_pixel_cooordinate' ,
                                                         'MODALITY_DESCRIPTION'  : 'Modality' ,
                                                      'STUDY_DESCRIPTION' : 'study_description' ,
                                                      'HARMONIZED_STUDY_DESCRIPTION' :'Imaging_Method' ,
                                                      'BODY_PART' : 'Anatomical_Region' ,
                                                      'LOCATION' : 'Location' ,
                                                      'LATERALITY' : 'Laterality' ,
                                                      'LN STATION' : 'Station' ,
                                                      'REMARKS' : 'Comments' ,
                                                      'NL_TUMOR_STATUS' : 'NL_Tumor_state' ,
                                                      'ORIENTATION' : 'Orientation'} 
                                                    )

In [ ]:
task_details_df.drop(columns = [  'Tagger', 'Unit' , 'InstanceUID'  , 'FAILURE_TYPE'] 
                     , inplace = True)

In [ ]:
task_details_df.info()

In [ ]:
task_details_df

In [ ]:
import pandas as pd
import numpy as np

def process_radiology_tasks(df):
    # ----------------------------------------------------
    # STEP 1: Process Dimensions and Diameter Fallback
    # ----------------------------------------------------
    df['Length'] = pd.to_numeric(df['Length'], errors='coerce')
    df['Width'] = pd.to_numeric(df['Width'], errors='coerce')

    # Identify if the row represents a lymph node
    is_node = df['LESION_DESCRIPTION'].astype(str).str.contains('node', case=False, na=False)

    # Core logic flag: If it's a node AND width exists, use Width. Otherwise, use Length.
    df['Diameter'] = np.where(is_node & df['Width'].notna(), df['Width'], df['Length'])
    
    # Securely fill any completely empty measurements with string 'NA' instead of a broken math zero
   

    # ----------------------------------------------------
    # STEP 2: The 3-Way NE Reason Extraction Logic
    # ----------------------------------------------------
    def extract_reason(val):
        val = str(val).strip().upper()
        
        if 'NE-' in val:
            # Splits at 'NE-' and extracts everything to the right of it
            return val.split('NE-', 1)[1].strip()
        return np.nan

    # Apply the rules individually to each specific column tracking category
    tl_reasons = df['TL_RESPONSE_CAT'].apply(extract_reason)
    ntl_reasons = df['NTL_RESPONSE'].apply(extract_reason)
    nl_reasons = df['NL_RESPONSE'].apply(extract_reason)

    # Chain them sequentially so they safely fall back without overwriting: 
    # Target Lesions -> Non-Target Lesions -> New Lesions -> 'NA'
    df['NE_Reason'] = tl_reasons.fillna(ntl_reasons).fillna(nl_reasons).fillna('NA')

    df.loc[df['NE_Reason'] != 'NA', 'Diameter'] = 'NE'
    
   # ----------------------------------------------------
    # STEP 3: Cleanup and Drop Source Fields Safely
    # ----------------------------------------------------
    cols_to_drop = ['LESION_DESCRIPTION', 'Length', 'Width', 'TL_RESPONSE_CAT', 'NTL_RESPONSE', 'NL_RESPONSE']
    existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
    df = df.drop(columns=existing_cols_to_drop)

    return df

# Execute the updated function 
task_details_final_df = process_radiology_tasks(task_details_df)

In [ ]:
task_details_final_df

In [ ]:
radiology_annotations_df.info()

In [ ]:
ANATOMICAL_KEYWORDS = [
    'RUL', 'LUL', 'RML', 'RLL', 'LLL', 'Lobe', 'Lung', 
    'Hilar', 'Mediastinal', 'Liver', 'Adrenal', 'Bone', 
    'Pleura', 'Node', 'Mass', 'Nodule'
]

def extract_other_location(row):
    location = str(row['Location']).strip().lower()
    comment = str(row['Comments'])
    other_location = str(row['LOCATION-OTHER'])   # the LOCATION-OTHER column

    if location == 'other' or location == 'nan' or location == "":

        found_terms = []

        # combine both fields so the keyword search covers Comments + Location-Other
        text_to_search = f"{comment} {other_location}"

        for term in ANATOMICAL_KEYWORDS:
            if re.search(rf'\b{term}\b', text_to_search, re.IGNORECASE):
                found_terms.append(term)

        if found_terms:
            return ", ".join(found_terms)

    return 'NA'

task_details_final_df['Other_location'] = task_details_final_df.apply(extract_other_location, axis=1)

In [ ]:
task_details_final_df.head()

In [ ]:
lobe_map = {
    'RUL - Apical': 'Right Upper Lobe',
    'RUL - Posterior': 'Right Upper Lobe',
    'RUL - Anterior': 'Right Upper Lobe',
    'RML - Lateral': 'Right Middle Lobe',
    'RML - Medial': 'Right Middle Lobe',
    'RLL - Superior': 'Right Lower Lobe',
    'RLL - Medial basal': 'Right Lower Lobe',
    'RLL - Anterior': 'Right Lower Lobe',
    'RLL - Lateral basal': 'Right Lower Lobe',
    'RLL - Posterior basal': 'Right Lower Lobe',
    'LUL - Apical': 'Left Upper Lobe',
    'LUL - Posterior': 'Left Upper Lobe',
    'LUL - Anterior': 'Left Upper Lobe',
    'LUL - Superior lingular': 'Left Upper Lobe',
    'LUL - Inferior lingular': 'Left Upper Lobe',
    'LLL - Superior': 'Left Lower Lobe',
    'LLL - Anteromedial basal': 'Left Lower Lobe',
    'LLL - Lateral basal': 'Left Lower Lobe',
    'LLL - Posterior basal': 'Left Lower Lobe'
}
# Standardize the Location column first by stripping whitespace
# (This handles the 'RUL - Posterior' line breaks in the image)
task_details_final_df['Location'] = task_details_final_df['Location'].str.replace('\n', ' ').str.strip()

task_details_final_df['Lobe'] = task_details_final_df['Location'].map(lobe_map)

In [ ]:
task_details_final_df[['Location', 'Lobe']].head()

In [ ]:
def clean_imaging_method(val):
    if pd.isna(val) or str(val).strip() == "":
        return val
    # and remove leading/trailing whitespace
    clean_val = str(val).replace('\n', ' ').strip()

    return re.sub(r'^[A-Z]{2,}\s+', '', clean_val)

task_details_final_df['Imaging_Method'] = task_details_final_df['Imaging_Method'].apply(clean_imaging_method)

In [ ]:
task_details_final_df['lesion_type'] = 'NTL'
task_details_final_df.loc[task_details_final_df['Lesion_id'].str.contains('^TL', case=False, na=False), 'lesion_type'] = 'TL'
task_details_final_df.loc[task_details_final_df['Lesion_id'].str.contains('^NL', case=False, na=False), 'lesion_type'] = 'NL'

In [ ]:
task_details_final_df = task_details_final_df[['PERSON_ID', 'study_id', 'series_id', 'StudyDate', 'Modality',
                                               'study_description', 'Imaging_Method', 'Anatomical_Region', 'series_type','lesion_type',
                                               'Lesion_id', 'Location', 'Other_location', 'Laterality', 'Lobe', 'Orientation', 
                                               'Station', 'Diameter', 'NL_Tumor_state', 'NE_Reason', 
                                               'Lesion_pixel_cooordinate', 'Comments']]

exclude_col = 'Diameter' 

target_cols = [col for col in task_details_final_df.columns if col != exclude_col]


task_details_final_df[target_cols] = task_details_final_df[target_cols].fillna('NA')

In [ ]:
task_details_final_df.info()

In [ ]:
columns_to_check = [col for col in task_details_final_df.columns if col != 'Lesion_pixel_cooordinate']

# 2. Drop duplicates based ONLY on those columns
task_details_final_df = task_details_final_df.drop_duplicates(subset=columns_to_check).reset_index(drop=True).copy()

In [ ]:
import ast
import math


def clean_coordinates(x):
    # Check for empty or basic null inputs up front
    if x == "N/A" or x is None:
        return x

    # Helper function to safely convert a single coordinate string/float
    def safe_int(coord):
        try:
            val = float(coord)
            if math.isnan(val):
                return None  # Or return 0 if you prefer a placeholder number
            return int(round(val))
        except:
            return coord

    # If it is already a list
    if isinstance(x, list):
        return [[safe_int(coord) for coord in sublist] for sublist in x]

    # If it is a string representing a list
    try:
        actual_list = ast.literal_eval(x)
        if isinstance(actual_list, list):
            return [
                [safe_int(coord) for coord in sublist] for sublist in actual_list
            ]
        return x
    except:
        return x

task_details_final_df['Lesion_pixel_cooordinate'] = task_details_final_df['Lesion_pixel_cooordinate'].apply(clean_coordinates)

In [ ]:
print(task_details_final_df['Lesion_pixel_cooordinate'])
print(task_details_final_df['Lesion_pixel_cooordinate'][0])
print(task_details_final_df['Lesion_pixel_cooordinate'][0][0])
print(task_details_final_df['Lesion_pixel_cooordinate'][0][0][0])
print(type(task_details_final_df['Lesion_pixel_cooordinate'][0][0][0]))

In [ ]:
task_details_final_df.sort_values(by = ['PERSON_ID', 'StudyDate', 'series_type' ], inplace = True)

In [ ]:
task_details_final_df.series_type.value_counts(dropna = False)

In [ ]:
task_details_final_df

In [ ]:
task_details_final_df.info()

In [ ]:
task_details_final_df['temp_sort_num'] = task_details_final_df['series_type'].str.extract(r'(\d+)').fillna(-1).astype(int)


task_details_final_df.sort_values(by=['PERSON_ID', 'temp_sort_num'], inplace=True)

# 3. Drop the temporary column so your dataframe stays clean
task_details_final_df.drop(columns=['temp_sort_num'], inplace=True)

In [ ]:
task_details_final_df['Diameter'] = task_details_final_df['Diameter'].fillna(0)

In [ ]:
task_details_final_df.to_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/RECIST_FINAL_OUTPUT/OUTPUT_CSVs/TASK_DETAILS_V13.csv', index= False)

# RECIST Response Calculations

In [ ]:
master_df_final.info()

In [ ]:
import numpy as np
import pandas as pd

def calculate_master_recist_v9_8(df):
    df = df.copy()

    # --- 1. CLEAN STRINGS SAFELY ---
    string_cols = ['SERIES_TYPE', 'LESION_ID', 'LESION_DESCRIPTION', 
                   'NTL_RESPONSE', 'NL_RESPONSE', 'TL_RESPONSE_CAT', 'REMARKS']
    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace(['nan', 'None', 'NaN', 'NAT', '<NA>', ''], np.nan)

    df['SERIES_TYPE'] = df['SERIES_TYPE'].str.upper()

    # --- 2. CLEAN NUMERIC COLUMNS ---
    for col in ['Length', 'Width']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # --- 3. ROBUST CATEGORIZATION ---
    df['CATEGORY'] = 'NTL'
    clean_lesion_ids = df['LESION_ID'].fillna('').astype(str).str.upper()
    df.loc[clean_lesion_ids.str.contains('^TL', na=False), 'CATEGORY'] = 'TL'
    df.loc[clean_lesion_ids.str.contains('^NL', na=False), 'CATEGORY'] = 'NL'

   
    def get_val_for_sod_v3(row):
        res_cat = str(row['TL_RESPONSE_CAT']).upper().strip() if pd.notna(row['TL_RESPONSE_CAT']) else ""
        desc = str(row['LESION_DESCRIPTION']).upper().strip() if pd.notna(row['LESION_DESCRIPTION']) else ""
        loc = str(row['LOCATION']).upper().strip() if pd.notna(row['LOCATION']) else ""
        stat = str(row['LN STATION']).upper().strip() if pd.notna(row['LN STATION']) else ""
    
   
        len_val = float(row['Length']) if (pd.notna(row['Length']) and float(row['Length']) > 0) else 0.0
        wid_val = float(row['Width']) if (pd.notna(row['Width']) and float(row['Width']) > 0) else 0.0
    
        # Standard Short Axis Diameter (SAD) selection rule: Width is preferred. Fallback to Length if Width is missing.
        sad_val = wid_val if wid_val > 0 else len_val
        
        
        node_keywords = ["NODE", "LYMPH"]
        
        is_node = (
            any(kw in desc for kw in node_keywords) or 
            ("STATION" in stat) or 
            (stat != "" and stat != "NAN" and stat != "NA") or 
            (loc in ["SUBCARINAL", "PARATRACHEAL"])
            )
        
        # =========================================================================
        # SPECIAL CONSIDERATIONS FOR LYMPH NODES
        # =========================================================================
        if is_node:
            
            # Condition 6: Node unequivocally not seen or metrics are entirely null/zero
            if sad_val == 0:
                return 0.0
                
            # Check if the node is obscured or faintly visible based on description string
            is_faint = 'FAINT' in desc
            is_clear = 'CLEAR' in desc or not is_faint # Default to clearly visible if not explicitly faint
            
            # Condition 1: Pathological Node (> 10 mm)
            if sad_val > 10.0:
                return sad_val
                
            # Condition 2 & 3: Node 5-10 mm or < 5 mm (Clearly Visible)
            if is_clear:
                # Record SAD exactly as it is and include in SOD calculation
                return sad_val
                
            # Condition 4 & 5: Node (Obscured or Faintly Visible - "FAINT")
            if is_faint:
                # If measurement drops below 5 mm, assign the default clinical baseline floor value of 5 mm
                if sad_val < 5.0:
                    return 5.0
                # If it is between 5 mm and 10 mm, take the measured value as it is
                elif 5.0 <= sad_val <= 10.0:
                    return sad_val
    
        # =========================================================================
        # NON-NODAL LESION FALLBACK RULES
        # =========================================================================
        if 'COMPLETE RESPONSE' in res_cat:
            return 0.0
            
        # Standard calculation logic for active target lesions
        if len_val > 0:
            return len_val
            
        return wid_val
    
   
    df['VAL_FOR_SOD'] = df.apply(get_val_for_sod_v3, axis=1)

    # --- 5. STUDY LEVEL AGGREGATION ---
    summary_list = []
    
    if 'StudyDate' in df.columns:
        df = df.sort_values(by=['PERSON_ID', 'StudyDate']).reset_index(drop=True)

    for (pid, tp), group in df.groupby(['PERSON_ID', 'SERIES_TYPE'], sort=False):
        tl_group = group[group['CATEGORY'] == 'TL']
        
        total_tls = len(tl_group)
        ne_tls = tl_group['TL_RESPONSE_CAT'].fillna('').astype(str).str.upper().str.contains('NE').sum()
        
        has_partial_ne = (ne_tls > 0 and ne_tls < total_tls)
        has_all_ne = (total_tls > 0 and ne_tls == total_tls)
        
        # Lock 'NE' as a string explicitly if any target lesion contains an NE status flag
        if has_all_ne or ne_tls > 0:
            sod = "NE"
            force_ne_flag = True
        else:
            sod = float(tl_group['VAL_FOR_SOD'].sum()) if not tl_group.empty else 0.0
            force_ne_flag = False

        tl_has_ne = tl_group['TL_RESPONSE_CAT'].dropna().astype(str).str.upper().str.contains('NE', na=False).any()
        tl_ne_reason = next((x for x in tl_group['TL_RESPONSE_CAT'].dropna() if 'NE' in str(x).upper()), 'NA')
        
        ntl_group = group[group['CATEGORY'] == 'NTL']
        ntl_ne_reason = next((x for x in ntl_group['NTL_RESPONSE'].dropna() if 'NE' in str(x).upper()), 'NA')
        
        nl_group = group[group['CATEGORY'] == 'NL']
        nl_ne_reason = next((x for x in nl_group['NL_RESPONSE'].dropna() if 'NE' in str(x).upper()), 'NA')

        study_date = group['StudyDate'].iloc[0] if 'StudyDate' in group.columns else None
        remarks = next((x for x in group['REMARKS'].dropna() if str(x).strip() != ''), 'NA')

        if ntl_group.empty:
            ntl_overall = 'NA'
        else:
            ntl_status = ntl_group['NTL_RESPONSE'].astype(str).str.upper().replace(['NAN', 'NONE'], np.nan).dropna().unique()
            
            if tp == 'BASELINE':
                ntl_overall = 'NA'

            elif len(ntl_status) == 0:
                ntl_overall = 'NA' 
                
            elif 'PD' in ntl_status:
                ntl_overall = 'PD'

            elif any('NE' in str(x) for x in ntl_status):
                # any single NTL marked NE -> overall NTL response is NE
                ntl_overall = 'NE'
            elif len(ntl_status) > 0 and all(x == 'CR' for x in ntl_status):
                ntl_overall = 'CR'
            else:
                ntl_overall = 'NN'

        nl_status = nl_group['NL_RESPONSE'].astype(str).str.upper().replace(['NAN', 'NONE'], np.nan).dropna().unique()
        if 'YES' in nl_status:
            nl_overall = 'YES'
        elif 'NE' in nl_status:
            nl_overall = 'NE'
        else:
            nl_overall = 'NO'

        tl_desc_str = tl_group['LESION_DESCRIPTION'].fillna('').astype(str).str.upper()
        tl_id_str = tl_group['LESION_ID'].fillna('').astype(str).str.upper()
        is_node_mask = tl_desc_str.str.contains('NODE') | tl_id_str.str.contains('NODE')

        tl_nodes = tl_group[is_node_mask]
        all_nodes_cr = True if tl_nodes.empty else bool((tl_nodes['Width'].fillna(0).astype(float) < 10).all() & (tl_nodes['Length'].fillna(0).astype(float) < 10).all())

        tl_masses = tl_group[~is_node_mask]
        all_masses_cr = True if tl_masses.empty else bool((tl_masses['VAL_FOR_SOD'].fillna(0).astype(float) == 0).all())

        # NEW: flag when EVERY target lesion is marked 'Complete Response' in TL_RESPONSE_CAT.
        # SOD may still be > 0 (e.g. a nodal TL keeps its residual short-axis), but response is CR.
        tl_resp_cat_str = tl_group['TL_RESPONSE_CAT'].fillna('').astype(str).str.upper().str.strip()
        all_tl_cr_cat = bool(total_tls > 0 and tl_resp_cat_str.str.contains('COMPLETE RESPONSE').all())

        summary_list.append({
            'PERSON_ID': pid, 'Study_date': study_date, 'SERIES_TYPE': tp, 'SOD': sod,
            'TL_HAS_NE': bool(tl_has_ne), 'IS_ALL_NE': force_ne_flag, 'IS_PARTIAL_NE': has_partial_ne,
            'TL_NE_Reason': tl_ne_reason, 'NTL_NE_Reason': ntl_ne_reason, 'NL_NE_Reason': nl_ne_reason,
            'NTL_OVERALL_RESPONSE': ntl_overall, 'NL_OVERALL_RESPONSE': nl_overall,
            'ALL_NODES_CR': all_nodes_cr, 'ALL_MASTES_CR': all_masses_cr, 'ALL_TL_CR_CAT': all_tl_cr_cat,
            'NO_TLS': bool(total_tls == 0), 'Remarks': remarks
        })

    if not summary_list:
        return pd.DataFrame(), df

    summary_df = pd.DataFrame(summary_list)

    # --- 5b. LOOKBACK REFERENCE & LOGIC FIXED FOR LAST VALID MINIMUM ---
    baseline_df = summary_df[summary_df['SERIES_TYPE'] == 'BASELINE']
    baseline_map = dict(zip(baseline_df['PERSON_ID'], baseline_df['SOD']))
    summary_df['BASELINE_SOD'] = summary_df['PERSON_ID'].map(baseline_map)

    nadir_list = []
    prev_cr_flag_list = []
    
    last_valid_nonzero_nadir = {}
    had_cr_map = {}

    for idx, row in summary_df.iterrows():
        pid = row['PERSON_ID']
        current_sod = row['SOD']
        
        if pid not in last_valid_nonzero_nadir:
            last_valid_nonzero_nadir[pid] = float(current_sod) if (current_sod != "NE" and float(current_sod) > 0) else np.nan
            had_cr_map[pid] = False
        
        prev_cr_flag_list.append(had_cr_map[pid])

        if current_sod == "NE" or row['IS_ALL_NE']:
            current_nadir = last_valid_nonzero_nadir[pid]
        else:
            val_numeric = float(current_sod)
            
            if val_numeric == 0.0:
                current_nadir = 0.0
                had_cr_map[pid] = True
            else:
                if had_cr_map[pid]:
                    current_nadir = last_valid_nonzero_nadir[pid]
                    had_cr_map[pid] = False
                elif np.isnan(last_valid_nonzero_nadir[pid]):
                    current_nadir = val_numeric
                    last_valid_nonzero_nadir[pid] = val_numeric
                else:
                    current_nadir = min(last_valid_nonzero_nadir[pid], val_numeric)
                    last_valid_nonzero_nadir[pid] = current_nadir

        nadir_list.append(current_nadir)

    summary_df['NADIR'] = nadir_list
    summary_df['PREVIOUS_TL_CR'] = prev_cr_flag_list

    # --- 6. TIMEPOINT RESPONSE GENERATOR ---
    def evaluate(row):
        is_baseline = row['SERIES_TYPE'] == 'BASELINE'
        ntl, nl = row['NTL_OVERALL_RESPONSE'], row['NL_OVERALL_RESPONSE']

        if row['SOD'] == "NE" or row['IS_ALL_NE']:
            s = "NE"
            tl_res = 'NE'
        else:
            s = float(row['SOD']) if pd.notna(row['SOD']) else 0.0
            tl_res = None
            
        b = float(row['BASELINE_SOD']) if (pd.notna(row['BASELINE_SOD']) and row['BASELINE_SOD'] != "NE") else 0.0
        n = float(row['NADIR']) if pd.notna(row['NADIR']) else 0.0

        if is_baseline:
            return pd.Series(['BASELINE', ntl, nl, 0.0, 0.0, 'NA'])

        has_valid_nadir = pd.notna(row['NADIR']) and n > 0



        if row['NO_TLS']:
            # no target lesions for this patient -> TL response is not applicable
            tl_res = 'NA'

        elif tl_res != 'NE':
            if row['TL_HAS_NE']:
                tl_res = 'NE'
            elif row['ALL_TL_CR_CAT']:
                # every target lesion is 'Complete Response' -> CR even if SOD > 0 (residual nodal SAD)
                tl_res = 'CR'
            elif row['ALL_MASTES_CR'] and row['ALL_NODES_CR']:
                tl_res = 'CR'
            elif has_valid_nadir and n > 0 and ((s - n) / n) >= 0.20 and (s - n) >= 5:
                tl_res = 'PD'
            elif b > 0 and ((s - b) / b) <= -0.30:
                tl_res = 'PR'
            else:
                tl_res = 'SD'

        if row['PREVIOUS_TL_CR'] and s != "NE" and float(s) > 0:
            tl_res = 'PD'

        if tl_res == 'PD':
            abs_chg = (s - n) if (has_valid_nadir and s != "NE") else 0.0
            pct_chg = (abs_chg / n * 100) if (has_valid_nadir and n > 0) else 0.0
        else:
            abs_chg = (s - b) if (b > 0 and s != "NE") else 0.0
            pct_chg = (abs_chg / b * 100) if (b > 0 and b > 0) else 0.0

        if nl == 'YES' or ntl == 'PD' or tl_res == 'PD':
            tp_res = 'PD'
        elif tl_res == 'PR' and ntl == 'CR' and nl == 'NO':
            tp_res = 'PR'
        elif tl_res == 'CR' and ntl == 'CR':
            tp_res = 'CR'
        elif tl_res in ['CR', 'PR'] and ntl in ['NN', 'NE']:
            tp_res = 'PR'
        elif tl_res == 'SD' and ntl in ['NN', 'NE', 'CR']:
            tp_res = 'SD'
        elif tl_res == 'NE':
            tp_res = 'NE'
        else:
            tp_res = tl_res

        return pd.Series([tl_res, ntl, nl, abs_chg, pct_chg, tp_res])

    res_cols = ['TL_OVERALL_RESPONSE', 'NTL_OVERALL_RESPONSE', 'NL_OVERALL_RESPONSE',
                'ABSOLUTE_CHANGE', 'PERCENT_CHANGE', 'OVERALL_RESPONSE']
    
    summary_df[res_cols] = summary_df.apply(evaluate, axis=1)

    # Re-apply explicit string preservation over the final Pandas matrix structures
    summary_df.loc[summary_df['IS_ALL_NE'] == True, 'SOD'] = "NE"

    # --- NEW: whenever TL_OVERALL_RESPONSE == 'NE', force SOD to "NE" as well ---
    summary_df.loc[summary_df['TL_OVERALL_RESPONSE'] == 'NE', 'SOD'] = "NE"

    final_ordered_cols = [
        'PERSON_ID', 'Study_date', 'SERIES_TYPE', 'SOD', 'NADIR',
        'TL_OVERALL_RESPONSE', 'TL_NE_Reason',
        'NTL_OVERALL_RESPONSE', 'NTL_NE_Reason',
        'NL_OVERALL_RESPONSE', 'NL_NE_Reason',
        'ABSOLUTE_CHANGE', 'PERCENT_CHANGE', 'OVERALL_RESPONSE', 'Remarks'
    ]

    return summary_df[final_ordered_cols], df

In [ ]:
report_df, master_df_final = calculate_master_recist_v9_8(master_df_final)

In [ ]:
report_df

In [ ]:
master_df_final.info()

In [ ]:
master_df_final.NL_RESPONSE.value_counts(dropna = False)

In [ ]:
master_df_final.info()

### Failure Type Calculations

In [ ]:
import pandas as pd
import re
import warnings


def _norm_key(name):
    return re.sub(r'[\s\-_]+', '_', str(name).strip().upper()).strip('_')


def _build_colmap(df):
    return {_norm_key(c): c for c in df.columns}


def _resolve_col(colmap, *candidates):
    for cand in candidates:
        key = _norm_key(cand)
        if key in colmap:
            return colmap[key]
    return None


def _get(row, colmap, *candidates, default=''):
    col = _resolve_col(colmap, *candidates)
    if col is None or col not in row.index:
        return default
    val = row[col]
    if pd.isna(val):
        return default
    return val


def _norm_series_value(s):
    return str(s).upper().strip().replace(" ", "")


def _resolve_category(row, colmap):
    cat = str(_get(row, colmap, 'CATEGORY')).upper().strip().replace(" ", "")
    if cat in ('TL', 'NTL', 'NL'):
        return cat
    lid = str(_get(row, colmap, 'LESION_ID', 'LESION ID')).upper().strip()
    if lid.startswith('NTL'):
        return 'NTL'
    if lid.startswith('NL'):
        return 'NL'
    if lid.startswith('TL'):
        return 'TL'
    return cat


def _get_failure_type(row, colmap):
    for cand in ('FAILURE TYPES', 'FAILURE_TYPES',
                 'FAILURE_TYPE', 'FAILURE TYPE'):
        val = _get(row, colmap, cand, default='')
        s = str(val).strip()
        if s and s.upper() != 'NAN':
            return s.upper()
    return ''


def analyze_failure_details_v16(report_df, raw_df):
    results = []

    raw = raw_df.copy()
    raw_colmap = _build_colmap(raw)
    report_colmap = _build_colmap(report_df)

    c_person = _resolve_col(raw_colmap, 'PERSON_ID', 'PERSON ID')
    c_series = _resolve_col(raw_colmap, 'SERIES_TYPE', 'SERIES TYPE')
    c_lesion = _resolve_col(raw_colmap, 'LESION_ID', 'LESION ID')
    c_val = _resolve_col(raw_colmap, 'VAL_FOR_SOD', 'VAL FOR SOD', 'LENGTH')
    c_loc = _resolve_col(raw_colmap, 'LOCATION')
    # NEW (v13): other-location column (raw 'LOCATION-OTHER' or derived 'Other_location')
    c_other = _resolve_col(raw_colmap, 'LOCATION-OTHER', 'LOCATION_OTHER',
                           'OTHER_LOCATION', 'Other_location', 'OTHER LOCATION')

    critical = {'PERSON_ID': c_person, 'SERIES_TYPE': c_series,
                'LESION_ID': c_lesion, 'VAL_FOR_SOD': c_val}
    missing = [name for name, col in critical.items() if col is None]
    if missing:
        warnings.warn(
            "analyze_failure_details: could not resolve column(s) "
            f"{missing}. Available raw columns: {list(raw.columns)}",
            stacklevel=2,
        )

    raw['__SERIES_NORM__'] = raw[c_series].map(_norm_series_value) if c_series else ''
    raw['__CAT__'] = raw.apply(lambda r: _resolve_category(r, raw_colmap), axis=1)

    per_lesion_nadir_tracker = {}
    if c_person and c_lesion and c_val:
        baseline_rows = raw[raw['__SERIES_NORM__'] == 'BASELINE']
        if not baseline_rows.empty:
            bl_aggregated = (baseline_rows
                             .groupby([c_person, c_lesion])[c_val]
                             .sum().to_dict())
            for (pid, l_id), val in bl_aggregated.items():
                per_lesion_nadir_tracker[(pid, l_id)] = val

    for idx, report_row in report_df.iterrows():
        pid = _get(report_row, report_colmap, 'PERSON_ID', 'PERSON ID')
        tp_raw = str(_get(report_row, report_colmap, 'SERIES_TYPE', 'SERIES TYPE'))
        tp = _norm_series_value(tp_raw)

        lrf_status, df_status = 'no', 'no'
        lrf_list, df_list = [], []
        unique_locations = []

        if tp == 'BASELINE':
            results.append({
                'LRF': 'no', 'LRF_Lesions_List': 'NA',
                'DF': 'no', 'DF_Lesions_List': 'NA',
                'Anatomical_location_of_the_PoF': 'NA',
            })
            continue

        patient_raw = raw[raw[c_person] == pid] if c_person else raw.iloc[0:0]
        group = patient_raw[patient_raw['__SERIES_NORM__'] == tp]

        def _loc_map(frame):
            m = {}
            if c_lesion and c_loc:
                for _, r in frame.iterrows():
                    lv = r[c_loc]
                    if pd.notna(lv) and str(lv).strip() and str(lv).strip().upper() != 'NAN':
                        loc_str = str(lv).strip()
                        # NEW (v13): if LOCATION-OTHER holds a real value, append it as
                        # 'Location-Other_location' (e.g. 'Other*-Right lower chest wall').
                        # If it's absent/NA, keep just the location.
                        if c_other is not None:
                            ov = r[c_other]
                            if pd.notna(ov):
                                ov_str = str(ov).strip()
                                if ov_str and ov_str.upper() not in ('NAN', 'NONE', 'NA', ''):
                                    loc_str = f"{loc_str}-{ov_str}"
                        m[str(r[c_lesion]).strip().upper()] = loc_str
            return m

        loc_current = _loc_map(group)
        loc_any = _loc_map(patient_raw)

        if str(_get(report_row, report_colmap, 'NL_OVERALL_RESPONSE')).upper().strip() == 'YES':
            nl_source = group if not group.empty else patient_raw
            nl_group = nl_source[nl_source['__CAT__'] == 'NL']
            for _, row in nl_group.iterrows():
                if str(_get(row, raw_colmap, 'NL_RESPONSE')).upper().strip() == 'YES':
                    f_type = _get_failure_type(row, raw_colmap)
                    l_id = str(_get(row, raw_colmap, 'LESION_ID', 'LESION ID')).strip()
                    if 'BOTH' in f_type:
                        lrf_status, df_status = 'yes', 'yes'
                        lrf_list.append(l_id)
                        df_list.append(l_id)
                    elif 'DF' in f_type:
                        df_status = 'yes'
                        df_list.append(l_id)
                    elif 'LRF' in f_type:
                        lrf_status = 'yes'
                        lrf_list.append(l_id)

        if str(_get(report_row, report_colmap, 'NTL_OVERALL_RESPONSE')).upper().strip() == 'PD':
            ntl_source = group if not group.empty else patient_raw
            ntl_group = ntl_source[ntl_source['__CAT__'] == 'NTL']
            for _, row in ntl_group.iterrows():
                if str(_get(row, raw_colmap, 'NTL_RESPONSE')).upper().strip() == 'PD':
                    f_type = _get_failure_type(row, raw_colmap)
                    l_id = str(_get(row, raw_colmap, 'LESION_ID', 'LESION ID')).strip()
                    if 'BOTH' in f_type:
                        lrf_status, df_status = 'yes', 'yes'
                        lrf_list.append(l_id)
                        df_list.append(l_id)
                    elif 'DF' in f_type:
                        df_status = 'yes'
                        df_list.append(l_id)
                    elif 'LRF' in f_type:
                        lrf_status = 'yes'
                        lrf_list.append(l_id)

        tl_group = group[group['__CAT__'] == 'TL']
        aggregated_tl = {}
        if c_lesion and c_val and not tl_group.empty:
            aggregated_tl = tl_group.groupby(c_lesion)[c_val].sum().to_dict()

        tl_failure_map = {}
        for _, row in tl_group.iterrows():
            ft = _get_failure_type(row, raw_colmap)
            if ft:
                key = str(_get(row, raw_colmap, 'LESION_ID', 'LESION ID')).strip().upper()
                tl_failure_map[key] = ft

        if str(_get(report_row, report_colmap, 'TL_OVERALL_RESPONSE')).upper().strip() == 'PD':
            # v16: LESION-LEVEL PD per SOP 4.1.1 -- a target lesion is PD when
            #     (current - nadir)/nadir >= 0.20  AND  (current - nadir) >= 5 mm,
            # where nadir = that lesion's lowest value since baseline.
            # EVERY target lesion that is individually PD is added; all tagged LRF.
            for l_id, current_val in aggregated_tl.items():
                lesion_nadir = per_lesion_nadir_tracker.get((pid, l_id), None)
                if lesion_nadir is not None and float(lesion_nadir) > 0:
                    increase = float(current_val) - float(lesion_nadir)
                    pct = increase / float(lesion_nadir)
                    if pct >= 0.20:
                        lrf_status = 'yes'
                        lrf_list.append(str(l_id).strip())

        # update per-lesion nadir AFTER the check; ignore non-positive values so that
        # NE timepoints (VAL_FOR_SOD = 0) or disappeared lesions don't corrupt the nadir.
        for l_id, current_val in aggregated_tl.items():
            cv = float(current_val)
            if cv > 0:
                if (pid, l_id) in per_lesion_nadir_tracker:
                    per_lesion_nadir_tracker[(pid, l_id)] = min(
                        per_lesion_nadir_tracker[(pid, l_id)], cv)
                else:
                    per_lesion_nadir_tracker[(pid, l_id)] = cv

        unique_lrf = sorted(set(lrf_list))
        unique_df = sorted(set(df_list))
        combined_failing_lesions = set(lrf_list + df_list)

        for l_id in combined_failing_lesions:
            key = str(l_id).strip().upper()
            loc = loc_current.get(key) or loc_any.get(key)
            if loc and str(loc).strip().upper() != 'NAN':
                unique_locations.append(str(loc).strip())

        unique_locations = sorted(set(unique_locations))

        results.append({
            'LRF': lrf_status,
            'LRF_Lesions_List': ", ".join(unique_lrf) if unique_lrf else "NA",
            'DF': df_status,
            'DF_Lesions_List': ", ".join(unique_df) if unique_df else "NA",
            'Anatomical_location_of_the_PoF': ", ".join(unique_locations) if unique_locations else "NA",
        })

    return pd.concat([report_df.reset_index(drop=True),
                      pd.DataFrame(results)], axis=1)


In [ ]:
recist_summary_df = analyze_failure_details_v12(report_df, master_df_final)

In [ ]:
master_df_final.info()

In [ ]:
recist_summary_df.drop(columns = ['PERCENT_CHANGE' , 'ABSOLUTE_CHANGE'], inplace = True)
recist_summary_df

In [ ]:
float_cols = ['SOD', 'NADIR']

for col in float_cols:
    def fmt(x):
        if isinstance(x, str) and x.upper() == 'NE':
            return 'NE'
        x_num = pd.to_numeric(x, errors='coerce')
        return f"{x_num:.4f}" if pd.notna(x_num) else x
    recist_summary_df[col] = recist_summary_df[col].apply(fmt)

In [ ]:
recist_summary_df['temp_sort_num'] = recist_summary_df['SERIES_TYPE'].str.extract(r'(\d+)').fillna(-1).astype(int)


recist_summary_df.sort_values(by=['PERSON_ID', 'temp_sort_num'], inplace=True)

# 3. Drop the temporary column so your dataframe stays clean
recist_summary_df.drop(columns=['temp_sort_num'], inplace=True)

In [ ]:
recist_summary_df.info()

In [ ]:
recist_summary_df.to_csv('/data/NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/RECIST_FINAL_OUTPUT/OUTPUT_CSVs/RECIST_SUMMARY_V13.csv', index= False)

In [ ]:
# NferenceInternalWorkingProject/shared/Nbs_V8/additional_nbs/RECIST_FINAL_OUTPUT/OUTPUT_CSVs/RECIST_SUMMARY.csv

In [ ]:
recist_summary_df.info()

In [ ]:
master_df_final.info()

In [ ]:
recist_summary_df.PERSON_ID.nunique()

# Verification

In [ ]:
task_details_final_df.info()

In [ ]:
recist_summary.PERSON_ID.nunique()

# Image Manifest & Mayo Cohort

In [ ]:
task_details_final_df.info()

In [ ]:
task_details_final_df.head()

In [ ]:
mayo_cohort_df.info()

In [ ]:
mayo_cohort_df.rename(columns = {'person_id': 'PERSON_ID'}, inplace = True)

In [ ]:
import pandas as pd


baseline_dates = task_details_final_df[
    task_details_final_df["series_type"].astype(str).str.upper() == "BASELINE"
][["PERSON_ID", "StudyDate"]].drop_duplicates()

# Ensure dates are in datetime format for comparison math
baseline_dates["StudyDate_dt"] = pd.to_datetime(baseline_dates["StudyDate"], errors="coerce")
mayo_cohort_df["crt_start_date_dt"] = pd.to_datetime(mayo_cohort_df["crt_start_date"], errors="coerce")


merged_dates = pd.merge(mayo_cohort_df, baseline_dates, on="PERSON_ID", how="left")


days_diff = (merged_dates["StudyDate_dt"] - merged_dates["crt_start_date_dt"]).dt.days.abs()


merged_dates["BASELINE_REFERENCE_DATE"] = merged_dates["StudyDate"].where(days_diff <= 28, None)


merged_dates["days_diff_val"] = days_diff.fillna(999)
merged_dates = merged_dates.sort_values("days_diff_val").drop_duplicates(subset=["PERSON_ID"], keep="first")

# Drop the helper columns we used for date arithmetic
final_joined_df = merged_dates.drop(columns=["StudyDate", "StudyDate_dt", "crt_start_date_dt", "days_diff_val"]).reset_index(drop = True)

# 4. Capitalize your requested specific column names
rename_dict = {
    
    "crt_start_date": "CRT_START_DATE",
    "crt_end_date": "CRT_END_DATE"
}

final_joined_df.columns = [rename_dict.get(col.lower(), col) for col in final_joined_df.columns]

# 5. Export directly out to your new CSV file footprint
final_joined_df.to_csv("final_mayo_cohort.csv", index=False)

# Preview the clean structural output
final_joined_df.head(10)

In [ ]:
final_joined_df.PERSON_ID.nunique()

In [ ]:
final_joined_df.BASELINE_REFERENCE_DATE.isnull().sum()

### Image manifest

In [ ]:
import pandas as pd


manifest_df = (
    task_details_final_df.groupby(["PERSON_ID", "StudyDate"], dropna=False)["series_id"]
    .nunique()
    .reset_index(name="image_count")
)

# 2. Save the final summary output directly to your requested CSV file name
manifest_df.to_csv("image_manifest.csv", index=False)

# View the structural format of your new manifest table
print(manifest_df.head(10))

In [ ]:
manifest_df.PERSON_ID.nunique()

In [ ]:
recist_summary_df.PERSON_ID.nunique()